# Train and compare the models on a free GPU (Colab / Kaggle)
**P8 — Multilingual Public Grievance Classifier and Router**

Runs the whole experiment: data -> baseline -> MuRIL, IndicBERT, mBERT -> comparison table, per-language scores,
confusion matrices -> a zip of results to download. About 15-30 minutes on a free T4 GPU for the default 1,000-row data.

**Before you start:** Runtime -> *Change runtime type* -> **T4 GPU** (Colab). On Kaggle: Settings -> Accelerator -> GPU, and turn **Internet** on.
Then run the cells from top to bottom (Runtime -> *Run all*).

> Reminder: the default data is team-generated (`label_source=generated`), so scores on its test split are inflated.
> Add real complaints in step 3 and keep some real text out of training for an honest evaluation.

## 1. Check the GPU

In [ ]:
import torch, subprocess
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout)
else:
    print("No GPU! Runtime -> Change runtime type -> GPU, then Run all again. (CPU training will be very slow.)")

## 2. Get the code
**Option A (default):** clone from GitHub (the repo must be public, and your latest work pushed).
**Option B:** if the clone fails, run the *upload* cell instead and choose `P8_public_grievance_classifier.zip`.

In [ ]:
REPO_URL = "https://github.com/sam131725/public_grievenes.git"   # change if your repo moved
BRANCH = "main"
import os, subprocess
if not os.path.exists("grievance-repo"):
    r = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, "grievance-repo"], capture_output=True, text=True)
    print(r.stdout or r.stderr)
%cd grievance-repo
!git log --oneline | head -3

In [ ]:
# OPTION B - only run this cell if the clone above failed
# from google.colab import files
# up = files.upload()                       # pick P8_public_grievance_classifier.zip
# import zipfile; zipfile.ZipFile(next(iter(up))).extractall(".")
# %cd grievance-classifier

## 3. Install packages and build the dataset

In [ ]:
!pip install -q transformers datasets scikit-learn pandas pyyaml langdetect sentencepiece
import sys; print(sys.version.split()[0])

In [ ]:
# Generated complaints (en/hi/mr). Real data: upload CSVs into data/raw/ first (see the next cell).
!python scripts/generate_complaints.py
!python scripts/prepare_data.py

In [ ]:
# OPTIONAL - add real complaints. Each CSV needs a text column (text/complaint/description) and a department column;
# urgency, sentiment and language are optional. Run this, then run the previous cell again.
# from google.colab import files
# up = files.upload()
# for name, data in up.items():
#     open(f"data/raw/{name}", "wb").write(data)

## 4. Baseline (TF-IDF + logistic regression)

In [ ]:
!python scripts/train_baseline.py

## 5. Fine-tune the three transformers
Each run saves `metrics.json`, `per_language_report.csv` and `confusion_matrix.csv` in `reports/<model>/`.
A failure in one model (for example a download problem) does not stop the others.

In [ ]:
import subprocess, time, json, pathlib
CONFIGS = ["muril", "indicbert", "mbert"]
EPOCHS = None   # e.g. 2 for a quick test; None uses the epochs in configs/*.yaml
status = {}
for name in CONFIGS:
    cmd = ["python", "scripts/train_transformer.py", "--config", f"configs/{name}.yaml"] + (["--epochs", str(EPOCHS)] if EPOCHS else [])
    print(f"\n===== {name} =====", flush=True)
    t0 = time.time()
    r = subprocess.run(cmd, text=True, capture_output=True)
    print(r.stdout[-1800:])
    if r.returncode != 0:
        print("FAILED:", r.stderr[-1500:])
    status[name] = {"ok": r.returncode == 0, "minutes": round((time.time() - t0) / 60, 1)}
print(status)

## 6. Optional ablation: class weighting off
Change one setting, keep everything else the same. Results go to `reports/muril_noweights/`.

In [ ]:
import yaml
cfg = yaml.safe_load(open("configs/muril.yaml"))
cfg["class_weighting"] = False
cfg["output_dir"] = "reports/muril_noweights"
yaml.safe_dump(cfg, open("configs/muril_noweights.yaml", "w"))
r = subprocess.run(["python", "scripts/train_transformer.py", "--config", "configs/muril_noweights.yaml"], text=True, capture_output=True)
print(r.stdout[-1200:], r.stderr[-800:] if r.returncode else "")

## 7. Comparison table

In [ ]:
!python scripts/compare_models.py

In [ ]:
import pandas as pd, matplotlib.pyplot as plt, numpy as np
from pathlib import Path
Path("docs/report/figures").mkdir(parents=True, exist_ok=True)
runs = {p.parent.name: p.parent for p in sorted(Path("reports").glob("*/metrics.json"))}

# per-language department macro-F1 (transformer runs write per_language_report.csv; baseline is at reports/)
frames = {}
for name, d in runs.items():
    f = d / "per_language_report.csv"
    if f.exists(): frames[name] = pd.read_csv(f).set_index("language")["macro_f1"]
if Path("reports/per_language_report.csv").exists() and "baseline" not in frames:
    frames["baseline"] = pd.read_csv("reports/per_language_report.csv").set_index("language")["macro_f1"]
pl = pd.DataFrame(frames).round(3)
print(pl.to_string())
pl.plot.bar(figsize=(7, 3.5), ylim=(0, 1.05), title="Department macro-F1 per language"); plt.ylabel("macro-F1"); plt.tight_layout()
plt.savefig("docs/report/figures/per_language_f1.png", dpi=160); plt.show()

In [ ]:
# confusion matrices
names = [n for n, d in runs.items() if (d / "confusion_matrix.csv").exists()]
if names:
    fig, axes = plt.subplots(1, len(names), figsize=(4.2 * len(names), 3.8), squeeze=False)
    for ax, n in zip(axes[0], names):
        cm = pd.read_csv(runs[n] / "confusion_matrix.csv", index_col=0)
        ax.imshow(cm.values, cmap="Blues"); ax.set_title(n)
        ax.set_xticks(range(cm.shape[1])); ax.set_xticklabels(cm.columns, rotation=40, ha="right")
        ax.set_yticks(range(cm.shape[0])); ax.set_yticklabels(cm.index)
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]): ax.text(j, i, cm.values[i, j], ha="center", va="center", fontsize=8)
    plt.tight_layout(); plt.savefig("docs/report/figures/confusion_matrices.png", dpi=160); plt.show()

## 8. Download the results
Downloads one zip with the metrics, CSVs, tables and figures (the large `model.pt` files are left out).
Copy the files into your repo's `reports/` and `docs/report/figures/`, then commit them.

In [ ]:
import shutil, os, glob
os.makedirs("download", exist_ok=True)
for f in glob.glob("reports/**/*", recursive=True):
    if os.path.isfile(f) and not f.endswith((".pt", ".bin", ".safetensors", ".pkl")) and "/tokenizer" not in f and "vocab" not in f and "sentencepiece" not in f:
        dst = os.path.join("download", f); os.makedirs(os.path.dirname(dst), exist_ok=True); shutil.copy(f, dst)
for f in glob.glob("docs/report/figures/*.png"):
    dst = os.path.join("download", f); os.makedirs(os.path.dirname(dst), exist_ok=True); shutil.copy(f, dst)
shutil.make_archive("grievance_results", "zip", "download")
try:
    from google.colab import files; files.download("grievance_results.zip")
except ImportError:
    print("Not on Colab: find grievance_results.zip in the working directory (Kaggle: Output tab).")

## Troubleshooting
- **No GPU / very slow:** Runtime -> Change runtime type -> T4 GPU.
- **Out of memory:** lower `batch_size` in `configs/*.yaml` (for example 8) or `max_len` to 96.
- **A model fails to download:** make sure Internet is on (Kaggle) and re-run only that cell; if IndicBERT keeps failing, report the error text.
- **Session disconnected:** Colab free sessions stop after a few hours idle or in total; re-run from the top. Download the results zip as soon as it is ready.
- **Stratified split error:** a class has too few rows; add more data in step 3.